# 07 — Fundamentals A/B at 120 Days: The Payoff Test

The return arc, established rigorously:
- 120-day cross-sectional return signal is **real** (honest IC ~0.09–0.10,
  non-overlapping t-stat significant, 11/11 sectors positive, 14/15 years positive).
- Short interest added **nothing**.
- Fundamentals are the **academically-supported** non-price signal (value/quality
  factors), now sourced PIT-clean from EDGAR with full 2010–2026 coverage.

**The question:** does adding value/quality fundamentals lift the validated 120-day
Ridge baseline? This is the experiment the whole non-price-data effort was building
toward, and the one most likely to work — value and quality are the most documented
return factors in the literature.

## Design

- **Horizon: 120 days**, cross-sectional rank — where the signal lives.
- **A/B:** `base` (price + macro) vs `base + fundamentals`. Identical folds/rows.
- **Both Ridge AND XGBoost.** Ridge won on price features, but fundamentals may carry
  *nonlinear* value (e.g. high-ROE-and-low-debt interactions) a tree captures — so we
  check both, not just Ridge.
- **grossmargin excluded** (72% null from EDGAR's inconsistent COGS tagging — low
  value, high missingness). The other 9 fundamental features are kept.
- **Missing fundamentals are median-imputed** (train-set medians only, to avoid
  leakage), so the full panel is usable and the A/B compares like-for-like rows.
- Metrics: rank IC and cost-aware net long-short spread, non-overlapping significance.

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, ttest_1samp
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent; break
else:
    raise RuntimeError('Run from inside the StockForecastRisk repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import (
    FEATURE_NAMES, NON_FEATURE_COLUMNS,
    FUNDAMENTAL_FEATURE_NAMES, FUNDAMENTALS_START,
)
from src.forecast_engine.data.loader import load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["symbol", "date"]).reset_index(drop=True)
print(f'Loaded {len(data):,} rows, {data["symbol"].nunique()} tickers, '
      f'{data["date"].min().date()} -> {data["date"].max().date()}')

In [ ]:
# Feature sets. grossmargin dropped (72% null); keep the other fundamentals.
NON_STATIONARY_LEVELS = ["sma_10","sma_20","sma_50","sma_200","ema_12","ema_26","vwap_20"]
SI = ["short_interest","short_interest_change","days_to_cover","days_to_cover_change"]
DROP_FUND = {"grossmargin"}
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | set(SI) | {"short_history","history_rows"}

fund_features = [c for c in FUNDAMENTAL_FEATURE_NAMES
                 if c not in DROP_FUND and c in data.columns and not data[c].isna().all()]
base_features = [c for c in FEATURE_NAMES
                 if c not in EXCLUDE and c not in FUNDAMENTAL_FEATURE_NAMES
                 and c in data.columns and not data[c].isna().all()]

HORIZON = 120
data[f"fwd_ret_{HORIZON}"] = data.groupby("symbol", group_keys=False).apply(
    lambda g: np.log(g["adj_close"].astype(float).shift(-HORIZON) / g["adj_close"].astype(float)))
target = f"fwd_ret_{HORIZON}"

print(f"base features      : {len(base_features)}")
print(f"+ fundamentals     : {len(fund_features)} -> {fund_features}")
print("fundamental null rates:")
display(data[fund_features].isna().mean().round(3).to_frame("null_frac"))

In [ ]:
# Model frame: require base features + target present. Fundamentals may be NaN and
# are imputed per-fold below, so we DON'T drop on them (that would throw away the
# pre-first-filing and untagged rows and bias the comparison).
model_data = data.dropna(subset=base_features + [target]).reset_index(drop=True)
model_data = model_data[model_data["date"] >= pd.Timestamp(FUNDAMENTALS_START)].reset_index(drop=True)
print(f"model rows (fundamentals era): {len(model_data):,}")

X_base = model_data[base_features].to_numpy(np.float32)
X_fund_raw = model_data[fund_features].to_numpy(np.float32)  # imputed per-fold
y = model_data[target].to_numpy(np.float32)
dates_all = model_data["date"].to_numpy()

## Harness — purged walk-forward, train-only imputation & scaling, IC & spread

In [ ]:
N_SPLITS = 5
def walk_forward_splits(frame, purge_days=HORIZON):
    dates = np.sort(frame["date"].unique())
    edges = np.array_split(dates, N_SPLITS + 1)
    for f in range(N_SPLITS):
        cutoff = edges[f][-1] - pd.Timedelta(days=purge_days * 2)
        tr = frame.index[frame["date"] <= cutoff]
        te = frame.index[frame["date"].isin(edges[f + 1])]
        if len(tr) and len(te):
            yield tr, te
folds = list(walk_forward_splits(model_data))

def daily_rank_ic(dates, y_true, y_pred):
    df = pd.DataFrame({"d": np.asarray(dates), "y": y_true, "p": y_pred})
    vals = [spearmanr(g["p"], g["y"]).correlation for _, g in df.groupby("d")
            if g["y"].nunique() > 2 and g["p"].nunique() > 2]
    vals = [v for v in vals if np.isfinite(v)]
    return (np.mean(vals) if vals else np.nan)

def net_spread(dates, y_true, y_pred, cost_bps=5.0):
    df = pd.DataFrame({"d": np.asarray(dates), "y": y_true, "p": y_pred}); nets=[]
    for _, g in df.groupby("d"):
        if len(g) < 20: continue
        hi = g["p"] >= g["p"].quantile(0.9); lo = g["p"] <= g["p"].quantile(0.1)
        if hi.sum() and lo.sum():
            nets.append((g.loc[hi,"y"].mean()-g.loc[lo,"y"].mean()) - 4*cost_bps/1e4)
    return float(np.mean(nets)) if nets else np.nan

def make_reg():
    return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=0, tree_method="hist")

In [ ]:
def run_arm(use_fund, model_kind):
    """One A/B arm. use_fund adds imputed fundamentals to the base features.
    Imputation uses TRAIN medians only (no leakage). Ridge scales; XGB doesn't need to."""
    rows = []
    for i, (tr, te) in enumerate(folds, 1):
        Xtr, Xte = X_base[tr], X_base[te]
        if use_fund:
            med = np.nanmedian(X_fund_raw[tr], axis=0)          # train-only medians
            ftr = np.where(np.isnan(X_fund_raw[tr]), med, X_fund_raw[tr])
            fte = np.where(np.isnan(X_fund_raw[te]), med, X_fund_raw[te])
            Xtr = np.hstack([Xtr, ftr]); Xte = np.hstack([Xte, fte])
        ytr = y[tr]; dte = dates_all[te]
        if model_kind == "ridge":
            sc = StandardScaler().fit(Xtr)
            pred = Ridge(alpha=1.0).fit(sc.transform(Xtr), ytr).predict(sc.transform(Xte))
        else:
            pred = make_reg().fit(Xtr, ytr).predict(Xte)
        rows.append({"fold": i, "features": "base+fund" if use_fund else "base",
                     "model": model_kind, "ic": daily_rank_ic(dte, y[te], pred),
                     "net": net_spread(dte, y[te], pred)})
    return pd.DataFrame(rows)

allres = pd.concat([run_arm(uf, mk) for uf in (False, True) for mk in ("ridge","xgboost")],
                   ignore_index=True)
summary = allres.groupby(["model","features"])[["ic","net"]].mean()
print("120-day A/B — base vs base+fundamentals:")
display(summary.round(5))

In [ ]:
# The lift table: does adding fundamentals raise IC / net spread?
for metric in ["ic","net"]:
    piv = allres.groupby(["model","features"])[metric].mean().unstack("features")
    piv["lift"] = piv["base+fund"] - piv["base"]
    print(f"\n{metric.upper()}:"); display(piv.round(5))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, metric in zip(axes, ["ic","net"]):
    for feats, style in [("base","-o"),("base+fund","--s")]:
        for model, color in [("ridge","tab:blue"),("xgboost","tab:orange")]:
            sub = allres[(allres.features==feats)&(allres.model==model)].sort_values("fold")
            ax.plot(sub.fold, sub[metric], style, color=color, alpha=0.8, label=f"{model} {feats}")
    ax.axhline(0, color="grey", lw=0.8); ax.set_title(f"120-day {metric} by fold"); ax.set_xlabel("fold")
axes[0].legend(fontsize=7); axes[1].axhline(0, color="red", ls="--")
plt.tight_layout(); plt.show()

## Non-overlapping significance of the lift

A fold-mean lift can be noise. This checks whether the *best* fundamentals arm beats
its base on independent (non-overlapping) 120-day periods — the honest test.

In [ ]:
# Pick the model with the larger IC lift, refit once over all folds to get per-date
# predictions for base and base+fund, then compare on non-overlapping periods.
best_model = (allres.groupby("model").apply(
    lambda d: d[d.features=="base+fund"].ic.mean() - d[d.features=="base"].ic.mean())
    ).idxmax()
print("model with larger fundamentals IC lift:", best_model)

def oos_preds(use_fund, model_kind):
    preds = np.full(len(model_data), np.nan, np.float32)
    for tr, te in folds:
        Xtr, Xte = X_base[tr], X_base[te]
        if use_fund:
            med = np.nanmedian(X_fund_raw[tr], axis=0)
            Xtr = np.hstack([Xtr, np.where(np.isnan(X_fund_raw[tr]), med, X_fund_raw[tr])])
            Xte = np.hstack([Xte, np.where(np.isnan(X_fund_raw[te]), med, X_fund_raw[te])])
        if model_kind == "ridge":
            sc = StandardScaler().fit(Xtr)
            preds[te] = Ridge(alpha=1.0).fit(sc.transform(Xtr), y[tr]).predict(sc.transform(Xte))
        else:
            preds[te] = make_reg().fit(Xtr, y[tr]).predict(Xte)
    return preds

def nonoverlap_ic(preds):
    df = model_data[["date"]].copy(); df["p"]=preds; df["y"]=y
    df = df.dropna(subset=["p"])
    ic_by_date = df.groupby("date").apply(
        lambda g: spearmanr(g["p"],g["y"]).correlation if g["y"].nunique()>2 and g["p"].nunique()>2 else np.nan
    ).dropna()
    ic_by_date.index = pd.to_datetime(ic_by_date.index)
    kept=[]; last=None
    for d in ic_by_date.index:
        if last is None or (d-last).days >= HORIZON*7/5:
            kept.append(d); last=d
    s = ic_by_date.loc[kept]
    t,p = ttest_1samp(s,0.0) if len(s)>1 else (np.nan,np.nan)
    return s.mean(), len(s), t, p

for label, uf in [("base", False), ("base+fund", True)]:
    m,n,t,p = nonoverlap_ic(oos_preds(uf, best_model))
    print(f"{label:10} non-overlap IC {m:+.4f}  (n={n}, t={t:+.2f}, p={p:.3f})")

## Conclusion — do fundamentals earn their place?

Fill from the numbers.

- **IC lift (base+fund − base):** Ridge ____ / XGBoost ____ — positive and consistent?
- **Net spread lift:** ____ — tradeable edge added after cost?
- **Nonlinearity:** did XGBoost gain more than Ridge (fundamentals carry interactions)?
- **Non-overlapping:** does the fundamentals arm beat base on independent periods?

### The decision

- **Meaningful, significant lift** → fundamentals earn their place. The return
  deliverable becomes a **120-day multi-signal model** (price + fundamentals), and
  this is the point to consider stacking further signals (analyst revisions). The
  value/quality literature is vindicated on your data.
- **No lift** → even the most-supported non-price signal doesn't beat price features
  at 120-day on this universe. That is a strong, publishable finding in itself: the
  120-day price signal is hard to improve on cheaply. Ship the 120-day price Ridge
  model as the return deliverable alongside the volatility engine, and treat further
  paid data as unjustified.

Either way the return arc is now complete and honestly evidenced: real at 120-day,
diversified, and tested for whether the best non-price signal improves it.